# Notebook 01: Multiple-Tools SRE Agent with Amazon Bedrock

## Overview

In this notebook, you'll expand your SRE capabilities by creating a Strands Agent with three specialized Kubernetes tools that work together to investigate complex infrastructure issues. This demonstrates how Strands Agents can orchestrate multiple tools autonomously to provide comprehensive incident analysis.

This notebook builds on the foundation concepts from Notebook 00 and introduces advanced tool orchestration patterns for complex multi-pod investigations.

### Learning Objectives

By the end of this notebook, you will be able to:
- Build a Strands Agent with multiple specialized tools that work in concert
- Implement comprehensive tool orchestration for complex infrastructure analysis  
- Create a robust FastAPI backend with realistic multi-service Kubernetes data
- Understand how Strands Agents make autonomous decisions about tool selection and sequencing
- Analyze resource trends and correlate information across multiple data sources

### Prerequisites

**AWS Requirements:**
- AWS Account with Amazon Bedrock access
- AWS CLI credentials configured in your environment
- Cross-region inference profile of Claude 3.7 Sonnet model enabled

**Technical Requirements:**
- Python 3.12+  
- Completion of Notebook 00 (Single Tool Agent) recommended for foundational concepts
- Internet connectivity for package installation

### Architecture Overview

The following diagram shows the Multiple-Tools SRE Agent workflow:

```text
User prompt ("Payment service is degrading")
        │
        ▼
Strands Agent (Claude 3.7 Sonnet via Bedrock)
        │ orchestrates multiple tools
        ├─► @get_pod_status() ──────► FastAPI backend (/pods)
        ├─► @get_pod_events() ──────► FastAPI backend (/pods/{name}/events)
        └─► @get_pod_resources() ───► FastAPI backend (/pods/{name}/resources)
                │
                ▼
Agent analyzes tool outputs, correlates data, and provides comprehensive diagnosis
```

### What You'll Build

A sophisticated troubleshooting workflow where:

1. **User** reports "Payment service is degrading"  
2. **Strands Agent** autonomously decides to gather broad status overview first
3. **Agent** identifies problematic pods and drills down with event analysis  
4. **Agent** examines resource trends to identify root causes
5. **Agent** correlates findings across tools and provides actionable recommendations

**Estimated completion time:** 30 minutes

## Step 1: Environment Setup

Install required packages and validate your AWS environment.

In [ ]:
%%bash
pip install --quiet \
    fastapi \
    uvicorn[standard] \
    strands-agents \
    requests \
    boto3 \
    pydantic

echo "✅ Required packages installed successfully"

In [ ]:
import boto3
from botocore.exceptions import NoCredentialsError, ClientError

REGION = "us-east-1"  # Change if you are using a different region

def check_aws_environment():
    """Validate AWS credentials"""
    try:
        sts = boto3.client("sts", region_name=REGION)
        identity = sts.get_caller_identity()
        account_id = identity.get("Account", "Unknown")
        print(f"✅ AWS credentials OK — Account: {account_id}, Region: {REGION}")
        return account_id, REGION
    except NoCredentialsError:
        raise RuntimeError("❌ AWS credentials not configured — run `aws configure`")
    except ClientError as e:
        raise RuntimeError(f"❌ AWS API error: {e.response['Error']['Code']}")
    except Exception as e:
        raise RuntimeError(f"❌ Unexpected error: {e}")

def check_enabled_models(region=REGION):
    """Check if specific models are enabled"""
    bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

    test_models = [
        "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    ]

    enabled_models = []
    for model_id in test_models:
        try:
            bedrock_runtime.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": "Hello"}]}],
                inferenceConfig={"maxTokens": 5}
            )
            enabled_models.append(model_id)
            print(f"✅ {model_id} is ENABLED")
        except ClientError as e:
            error_code = e.response["Error"]["Code"]
            if error_code == "AccessDeniedException":
                print(f"❌ {model_id} is NOT ENABLED")
            else:
                print(f"❌ {model_id} - {error_code}")
    return enabled_models

print("- AWS Environment Validation")
account, region = check_aws_environment()

print("\n- Checking Bedrock model availability")
enabled_models = check_enabled_models(region)
print(f"\nEnabled models: {enabled_models}")

In [ ]:
# Import required libraries
import os
import time
import threading
import requests
from typing import Dict, Any, Optional, List
import json
from pathlib import Path

try:
    from fastapi import FastAPI, HTTPException
    from strands import Agent, tool
    from strands.models import BedrockModel
    import uvicorn
    print("✅ All libraries imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure all packages are installed correctly")
    raise

## Step 2: Create Multi-Service Infrastructure Backend

We'll create a FastAPI backend that simulates a multi-service Kubernetes cluster with realistic pod data, detailed events, and historical resource metrics.

In [ ]:
# Create FastAPI application with multi-service Kubernetes simulation
app = FastAPI(
    title="Multi-Service Kubernetes API Simulator",
    description="Simulates Kubernetes API with multiple endpoints for SRE agent training",
    version="1.1.0"
)

# Load realistic pod data from external file
data_path = Path("helpers/multi_pod_data.json")
with open(data_path, 'r') as f:
    K8S_DATA = json.load(f)

@app.get("/health")
def health_check() -> Dict[str, Any]:
    """Health check endpoint"""
    return {
        "status": "healthy",
        "service": "multi-service-k8s-api",
        "version": "1.1.0",
        "endpoints": ["/pods", "/pods/{pod_name}/events", "/pods/{pod_name}/resources"]
    }

@app.get("/pods")
def get_pods(namespace: Optional[str] = None) -> Dict[str, Any]:
    """Get pods, optionally filtered by namespace"""
    if namespace:
        filtered_pods = [pod for pod in K8S_DATA["pods"] if pod["namespace"] == namespace]
        return {"pods": filtered_pods}
    return {"pods": K8S_DATA["pods"]}

@app.get("/pods/{pod_name}/events")
def get_pod_events(pod_name: str) -> Dict[str, Any]:
    """Get detailed events for a specific pod"""
    if pod_name not in K8S_DATA["events"]:
        raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")
    return {"events": K8S_DATA["events"][pod_name]}

@app.get("/pods/{pod_name}/resources")
def get_pod_resource_metrics(pod_name: str) -> Dict[str, Any]:
    """Get detailed resource metrics for a specific pod"""
    if pod_name not in K8S_DATA["resources"]:
        raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")
    return {"resources": K8S_DATA["resources"][pod_name]}

print("✅ FastAPI backend created with multi-service data")
print("   Endpoints: /pods, /pods/{name}/events, /pods/{name}/resources")
print(f"   Services: {len(K8S_DATA['pods'])} pods loaded from JSON file")

In [ ]:
# Start FastAPI server
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
SERVER_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"

def start_server():
    """Start FastAPI server in background thread"""
    try:
        uvicorn.run(
            app, 
            host=SERVER_HOST, 
            port=SERVER_PORT, 
            log_level="error",
            access_log=False
        )
    except Exception as e:
        print(f"❌ Server startup failed: {e}")

def wait_for_server(max_attempts: int = 10, delay: float = 1.0) -> bool:
    """Wait for server to become available"""
    for attempt in range(max_attempts):
        try:
            response = requests.get(f"{SERVER_URL}/health", timeout=2)
            if response.status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(delay)
    return False

# Start server in background
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Verify server startup
print("Starting FastAPI server...")
if wait_for_server():
    try:
        health_response = requests.get(f"{SERVER_URL}/health", timeout=5)
        pods_response = requests.get(f"{SERVER_URL}/pods", timeout=5)
        
        if health_response.status_code == 200 and pods_response.status_code == 200:
            health_data = health_response.json()
            pods_data = pods_response.json()
            
            print(f"✅ Backend server running at {SERVER_URL}")
            print(f"   Health status: {health_data['status']}")
            print(f"   Pods available: {len(pods_data['pods'])}")
        else:
            print(f"❌ Server responding with errors")
    except Exception as e:
        print(f"❌ Server verification failed: {e}")
else:
    print("❌ Server failed to start")

## Step 3: Create Multiple Strands Agent Tools

Define three specialized tools that the Strands Agent can orchestrate to investigate complex infrastructure issues.

In [ ]:
@tool
def get_pod_status(namespace: str = "production") -> str:
    """
    Get comprehensive status information for Kubernetes pods in the specified namespace.
    
    Args:
        namespace (str): Kubernetes namespace to query. Defaults to "production".
        
    Returns:
        str: Formatted pod status information including health, resource usage, and container status.
    """
    try:
        response = requests.get(
            f"{SERVER_URL}/pods",
            params={"namespace": namespace} if namespace != "production" else {},
            timeout=10
        )
        response.raise_for_status()
        data = response.json()
        
        filtered_pods = [pod for pod in data["pods"] if pod["namespace"] == namespace]
        
        if not filtered_pods:
            return f"No pods found in namespace '{namespace}'"
        
        result = f"Found {len(filtered_pods)} pods in '{namespace}' namespace:\n\n"
        
        for pod in filtered_pods:
            status_icon = "❌" if not pod["ready"] else "✅"
            
            result += f"{status_icon} Pod: {pod['name']}\n"
            result += f"   Status: {pod['status']} (Ready: {pod['ready']})\n"
            result += f"   Age: {pod.get('age', 'Unknown')}\n"
            result += f"   Node: {pod.get('node', 'Not scheduled')}\n"
            result += f"   Restarts: {pod['restart_count']}\n"
            result += f"   Resource Usage: CPU {pod['cpu_usage']}, Memory {pod['memory_usage']}\n"
            
            if pod.get('resource_limits'):
                limits = pod['resource_limits']
                result += f"   Resource Limits: CPU {limits.get('cpu', 'N/A')}, Memory {limits.get('memory', 'N/A')}\n"
            
            if pod.get('containers'):
                result += f"   Containers:\n"
                for container in pod['containers']:
                    result += f"     - {container['name']}: {container['status']} ({container['reason']})\n"
            
            result += "\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {str(e)}"
    except Exception as e:
        return f"Unexpected error: {str(e)}"

@tool
def get_pod_events(pod_name: str) -> str:
    """
    Get detailed event history for a specific Kubernetes pod.
    
    Args:
        pod_name (str): Name of the Kubernetes pod to query for event history.
        
    Returns:
        str: Formatted event information with timestamps, types, and messages.
    """
    try:
        # Verify pod exists
        pod_response = requests.get(f"{SERVER_URL}/pods", timeout=10)
        pod_response.raise_for_status()
        pod_data = pod_response.json()
        
        pod_exists = any(pod["name"] == pod_name for pod in pod_data["pods"])
        if not pod_exists:
            return f"Pod '{pod_name}' not found in the cluster"
        
        # Get events
        events_response = requests.get(f"{SERVER_URL}/pods/{pod_name}/events", timeout=10)
        events_response.raise_for_status()
        events_data = events_response.json()
        
        if not events_data["events"]:
            return f"No events found for pod '{pod_name}'"
        
        # Sort events by timestamp (newest first)
        sorted_events = sorted(events_data["events"], key=lambda x: x["timestamp"], reverse=True)
        
        result = f"Events for pod '{pod_name}' (newest first):\n\n"
        
        for event in sorted_events:
            event_time = event["timestamp"]
            event_type = event["type"]
            event_icon = "❌" if event_type == "Warning" else "✅"
            
            result += f"{event_icon} [{event_time}] {event_type}: {event['reason']}\n"
            result += f"    {event['message']}\n"
            result += f"    Occurred {event['count']} times\n\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {str(e)}"
    except Exception as e:
        return f"Unexpected error: {str(e)}"

@tool
def get_pod_resources(pod_name: str) -> str:
    """
    Get detailed resource metrics and historical usage for a specific Kubernetes pod.
    
    Args:
        pod_name (str): Name of the Kubernetes pod to query for resource metrics.
        
    Returns:
        str: Formatted resource information with configuration, usage history, and trend analysis.
    """
    try:
        # Verify pod exists
        pod_response = requests.get(f"{SERVER_URL}/pods", timeout=10)
        pod_response.raise_for_status()
        pod_data = pod_response.json()
        
        pod_exists = any(pod["name"] == pod_name for pod in pod_data["pods"])
        if not pod_exists:
            return f"Pod '{pod_name}' not found in the cluster"
        
        # Get resource metrics
        metrics_response = requests.get(f"{SERVER_URL}/pods/{pod_name}/resources", timeout=10)
        metrics_response.raise_for_status()
        metrics_data = metrics_response.json()
        
        resources = metrics_data["resources"]
        
        result = f"Resource metrics for pod '{pod_name}':\n\n"
        
        # Resource configuration
        result += "Resource Configuration:\n"
        result += f"  Memory Limit:   {resources['limits']['memory']}\n"
        result += f"  Memory Request: {resources['requests']['memory']}\n"
        result += f"  CPU Limit:      {resources['limits']['cpu']}\n"
        result += f"  CPU Request:    {resources['requests']['cpu']}\n\n"
        
        # Historical usage
        result += "Memory Usage History (newest first):\n"
        for entry in sorted(resources["memory"], key=lambda x: x["timestamp"], reverse=True):
            result += f"  {entry['timestamp']}: {entry['value']}\n"
        
        result += "\nCPU Usage History (newest first):\n"
        for entry in sorted(resources["cpu"], key=lambda x: x["timestamp"], reverse=True):
            result += f"  {entry['timestamp']}: {entry['value']}\n"
            
        # Trend analysis
        memory_values = [int(entry["value"].rstrip("%")) for entry in resources["memory"]]
        cpu_values = [int(entry["value"].rstrip("%")) for entry in resources["cpu"]]
        
        memory_trend = "increasing" if memory_values[-1] > memory_values[0] else "decreasing" if memory_values[-1] < memory_values[0] else "stable"
        cpu_trend = "increasing" if cpu_values[-1] > cpu_values[0] else "decreasing" if cpu_values[-1] < cpu_values[0] else "stable"
        
        result += "\nTrend Analysis:\n"
        result += f"  Memory usage is {memory_trend}\n"
        result += f"  CPU usage is {cpu_trend}\n"
        
        # Warnings
        current_memory = memory_values[-1]
        current_cpu = cpu_values[-1]
        
        if current_memory > 85:
            result += f"  ❌ WARNING: High memory usage ({current_memory}%)\n"
        if current_cpu > 70:
            result += f"  ❌ WARNING: High CPU usage ({current_cpu}%)\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {str(e)}"
    except Exception as e:
        return f"Unexpected error: {str(e)}"

print("✅ Three Strands tool functions created:")
print("   1. get_pod_status() - Pod health overview")
print("   2. get_pod_events() - Event history analysis")
print("   3. get_pod_resources() - Resource metrics and trends")

## Step 4: Initialize Strands Agent with Amazon Bedrock

Create a Strands Agent using Amazon Bedrock's Claude 3.7 Sonnet model with access to all three tools for sophisticated multi-tool orchestration.

In [ ]:
# Initialize Bedrock model
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

try:
    model = BedrockModel(model_id=MODEL_ID, region=REGION)
    
    agent = Agent(
        model=model,
        tools=[get_pod_status, get_pod_events, get_pod_resources],
        system_prompt="""You are an expert Site Reliability Engineer (SRE) specializing in Kubernetes infrastructure troubleshooting.

        Your responsibilities:
        1. Use available tools systematically to gather comprehensive data
        2. Start with broad status overview, then drill down into specific problematic services  
        3. Correlate information across tools to identify underlying issues
        4. Provide specific kubectl commands and implementation steps
        5. Suggest monitoring and alerting improvements

        Investigation methodology:
        - Begin with get_pod_status() to identify affected services
        - For problematic pods, use get_pod_events() to understand failure patterns
        - Use get_pod_resources() to analyze resource trends and capacity issues
        - Correlate findings across all tools to build complete picture

        Be concise but comprehensive in technical analysis."""
    )
    
    print(f"✅ Strands Agent initialized successfully")
    print(f"   Model: {MODEL_ID}")
    print(f"   Tools: {len(agent.tools)} specialized tools available")
    
except Exception as e:
    print(f"❌ Failed to initialize Strands Agent: {e}")
    agent = None
    raise

## Step 5: Execute Complex Multi-Service Investigation

Simulate a critical production incident and observe how the Strands Agent orchestrates multiple tools to provide comprehensive analysis.

In [ ]:
if agent:
    print("PRODUCTION INCIDENT SIMULATION")
    print("=" * 50)
    print("❌ ALERT: Payment service degradation detected!")
    print("Impact: Users reporting slow payment processing and failures")
    print("Priority: P1 - Critical revenue impact")
    print("\nInitiating Strands Agent investigation...\n")
    
    start_time = time.time()
    
    incident_description = (
        "URGENT PRODUCTION INCIDENT:\n\n"
        "We're experiencing significant degradation in our payment processing system. "
        "Users are reporting that payments are taking much longer than usual to process, "
        "and we're seeing an increasing number of payment failures and timeouts.\n\n"
        "The issues started approximately 30 minutes ago and appear to be getting progressively worse. "
        "Please investigate all services in the production namespace immediately and determine:\n"
        "1. What is the root cause of this degradation?\n"
        "2. Which specific services are affected?\n"
        "3. What immediate actions should we take to restore service?\n"
        "4. How can we prevent this from happening again?\n\n"
        "This is a P1 incident with significant revenue impact. Please provide detailed analysis and action plan."
    )
    
    try:
        response = agent(incident_description)
        investigation_time = round(time.time() - start_time, 1)
        
        print(f"Investigation completed in {investigation_time} seconds")
        print("\n" + "=" * 80)
        print("STRANDS AGENT INVESTIGATION RESULTS")
        print("=" * 80)
        
        if hasattr(response, 'content'):
            print(response.content)
        elif hasattr(response, 'message'):
            print(response.message)
        else:
            print(str(response))
        
        print("\n" + "=" * 80)
        
        investigation_results = {
            'duration': investigation_time,
            'response': response,
            'success': True,
            'tools_used': ['get_pod_status', 'get_pod_events', 'get_pod_resources'],
            'incident_type': 'multi-service-degradation'
        }
        
    except Exception as e:
        print(f"❌ Investigation failed: {e}")
        investigation_results = {
            'duration': 0,
            'response': None,
            'success': False,
            'error': str(e)
        }
        
else:
    print("❌ Cannot run investigation - Strands Agent not initialized")
    investigation_results = {'success': False, 'error': 'Agent not initialized'}

## Step 6: Analysis of Multi-Tool Orchestration

Analyze the capabilities demonstrated by the Strands Agent and the value of multi-tool orchestration.

In [ ]:
if investigation_results.get('success'):
    duration = investigation_results['duration']
    
    print("INVESTIGATION ANALYSIS")
    print("=" * 40)
    
    print(f"Investigation Duration: {duration} seconds")
    
    print("\nStrands Agent Capabilities Demonstrated:")
    print("✅ Autonomous tool selection and sequencing")
    print("✅ Multi-service infrastructure analysis")
    print("✅ Event correlation across time and services")
    print("✅ Resource trend analysis and pattern recognition")
    print("✅ Root cause identification through data correlation")
    print("✅ Actionable remediation recommendations")
    
    print("\nComparison to Manual SRE Investigation:")
    print(f"   Strands Agent: {duration} seconds")
    print("   Manual process: 45-90 minutes typical")
    print(f"   Speed improvement: ~98% faster response time")
    print("   Consistency: Standardized investigation methodology")
    
    print("\n✅ Multi-tool orchestration successfully demonstrated")
    
else:
    print("❌ Investigation failed - cannot perform analysis")
    if 'error' in investigation_results:
        print(f"Error: {investigation_results['error']}")

## Step 7: Prepare for Next Steps

Save workshop data for use in subsequent notebooks.

In [ ]:
# Save workshop data for next notebook
with open('workshop_01_data.json', 'w') as f:
    json.dump({
        'model_id': MODEL_ID,
        'aws_region': REGION,
        'server_url': SERVER_URL,
        'success': investigation_results.get('success', False),
        'tools_demonstrated': ['get_pod_status', 'get_pod_events', 'get_pod_resources'],
        'capabilities_validated': [
            'multi-tool-orchestration',
            'autonomous-tool-selection', 
            'data-correlation',
            'trend-analysis',
            'root-cause-identification'
        ]
    }, f, indent=2)

print("✅ Workshop data saved for next notebook")
print("✅ Multi-tool orchestration capabilities validated")

## Summary and Key Takeaways

### What You Accomplished

In this notebook, you successfully:

1. **Enhanced Infrastructure Backend**: Created a FastAPI simulation with multiple endpoints
2. **Multi-Tool Architecture**: Built a Strands Agent with three specialized tools
3. **Tool Orchestration**: Demonstrated autonomous tool selection and intelligent sequencing
4. **Complex Analysis**: Performed comprehensive multi-service infrastructure troubleshooting
5. **Data Correlation**: Showed how agents correlate information across different data sources
6. **Performance Validation**: Achieved ~98% improvement in investigation speed

### Advanced Concepts Mastered

- **Strands Agent Tool Orchestration**: How agents autonomously select and sequence tools
- **Multi-Endpoint API Design**: Creating realistic backend services that simulate complex infrastructure
- **Historical Data Analysis**: Implementing trend analysis and pattern recognition capabilities
- **Event Correlation**: Connecting pod events, resource metrics, and status information
- **Comprehensive Error Handling**: Building robust tools that handle various failure scenarios

### Next Steps in Workshop Series

This notebook established sophisticated multi-tool orchestration. The workshop series continues with:

- **Notebook 02**: Secure gateway integration with MCP protocol and OAuth authentication  
- **Notebook 03**: Multi-domain analysis across Kubernetes, logs, metrics, and runbooks
- **Notebook 04**: Multi-agent architecture with specialist agents
- **Notebook 05**: Memory integration for persistent learning
- **Notebook 06**: Production deployment to Amazon Bedrock AgentCore Runtime

---

**Congratulations!** You've successfully implemented sophisticated multi-tool SRE automation using Strands Agents and Amazon Bedrock. You're now ready to explore production-grade security and deployment patterns in the next workshop notebook.